# Subject: support
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/PC/layout/support

### File: appxmanifest.py

In [ ]:
"""
File generation for APPX/MSIX manifests.
"""

__author__ = "Steve Dower <steve.dower@python.org>"
__version__ = "3.8"


import ctypes
import io
import os

from pathlib import PureWindowsPath
from xml.etree import ElementTree as ET

from .constants import *

__all__ = ["get_appx_layout"]


APPX_DATA = dict(
    Name="PythonSoftwareFoundation.Python.{}".format(VER_DOT),
    Version="{}.{}.{}.0".format(VER_MAJOR, VER_MINOR, VER_FIELD3),
    Publisher=os.getenv(
        "APPX_DATA_PUBLISHER", "CN=4975D53F-AA7E-49A5-8B49-EA4FDC1BB66B"
    ),
    DisplayName="Python {}".format(VER_DOT),
    Description="The Python {} runtime and console.".format(VER_DOT),
)

APPX_PLATFORM_DATA = dict(
    _keys=("ProcessorArchitecture",),
    win32=("x86",),
    amd64=("x64",),
    arm32=("arm",),
    arm64=("arm64",),
)

PYTHON_VE_DATA = dict(
    DisplayName="Python {}".format(VER_DOT),
    Description="Python interactive console",
    Square150x150Logo="_resources/pythonx150.png",
    Square44x44Logo="_resources/pythonx44.png",
    BackgroundColor="transparent",
)

PYTHONW_VE_DATA = dict(
    DisplayName="Python {} (Windowed)".format(VER_DOT),
    Description="Python windowed app launcher",
    Square150x150Logo="_resources/pythonwx150.png",
    Square44x44Logo="_resources/pythonwx44.png",
    BackgroundColor="transparent",
    AppListEntry="none",
)

PIP_VE_DATA = dict(
    DisplayName="pip (Python {})".format(VER_DOT),
    Description="pip package manager for Python {}".format(VER_DOT),
    Square150x150Logo="_resources/pythonx150.png",
    Square44x44Logo="_resources/pythonx44.png",
    BackgroundColor="transparent",
    AppListEntry="none",
)

IDLE_VE_DATA = dict(
    DisplayName="IDLE (Python {})".format(VER_DOT),
    Description="IDLE editor for Python {}".format(VER_DOT),
    Square150x150Logo="_resources/idlex150.png",
    Square44x44Logo="_resources/idlex44.png",
    BackgroundColor="transparent",
)

PY_PNG = "_resources/py.png"

APPXMANIFEST_NS = {
    "": "http://schemas.microsoft.com/appx/manifest/foundation/windows10",
    "m": "http://schemas.microsoft.com/appx/manifest/foundation/windows10",
    "uap": "http://schemas.microsoft.com/appx/manifest/uap/windows10",
    "rescap": "http://schemas.microsoft.com/appx/manifest/foundation/windows10/restrictedcapabilities",
    "rescap4": "http://schemas.microsoft.com/appx/manifest/foundation/windows10/restrictedcapabilities/4",
    "desktop4": "http://schemas.microsoft.com/appx/manifest/desktop/windows10/4",
    "desktop6": "http://schemas.microsoft.com/appx/manifest/desktop/windows10/6",
    "uap3": "http://schemas.microsoft.com/appx/manifest/uap/windows10/3",
    "uap4": "http://schemas.microsoft.com/appx/manifest/uap/windows10/4",
    "uap5": "http://schemas.microsoft.com/appx/manifest/uap/windows10/5",
}

APPXMANIFEST_TEMPLATE = """<?xml version="1.0" encoding="utf-8"?>
<Package IgnorableNamespaces="desktop4 desktop6"
    xmlns="http://schemas.microsoft.com/appx/manifest/foundation/windows10"
    xmlns:uap="http://schemas.microsoft.com/appx/manifest/uap/windows10"
    xmlns:rescap="http://schemas.microsoft.com/appx/manifest/foundation/windows10/restrictedcapabilities"
    xmlns:rescap4="http://schemas.microsoft.com/appx/manifest/foundation/windows10/restrictedcapabilities/4"
    xmlns:desktop4="http://schemas.microsoft.com/appx/manifest/desktop/windows10/4"
    xmlns:uap4="http://schemas.microsoft.com/appx/manifest/uap/windows10/4"
    xmlns:uap5="http://schemas.microsoft.com/appx/manifest/uap/windows10/5">
    <Identity Name=""
              Version=""
              Publisher=""
              ProcessorArchitecture="" />
    <Properties>
        <DisplayName></DisplayName>
        <PublisherDisplayName>Python Software Foundation</PublisherDisplayName>
        <Description></Description>
        <Logo>_resources/pythonx50.png</Logo>
    </Properties>
    <Resources>
        <Resource Language="en-US" />
    </Resources>
    <Dependencies>
        <TargetDeviceFamily Name="Windows.Desktop" MinVersion="10.0.17763.0" MaxVersionTested="" />
    </Dependencies>
    <Capabilities>
        <rescap:Capability Name="runFullTrust"/>
    </Capabilities>
    <Applications>
    </Applications>
    <Extensions>
    </Extensions>
</Package>"""


RESOURCES_XML_TEMPLATE = r"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<!--This file is input for makepri.exe. It should be excluded from the final package.-->
<resources targetOsVersion="10.0.0" majorVersion="1">
    <packaging>
        <autoResourcePackage qualifier="Language"/>
        <autoResourcePackage qualifier="Scale"/>
        <autoResourcePackage qualifier="DXFeatureLevel"/>
    </packaging>
    <index root="\" startIndexAt="\">
        <default>
            <qualifier name="Language" value="en-US"/>
            <qualifier name="Contrast" value="standard"/>
            <qualifier name="Scale" value="100"/>
            <qualifier name="HomeRegion" value="001"/>
            <qualifier name="TargetSize" value="256"/>
            <qualifier name="LayoutDirection" value="LTR"/>
            <qualifier name="Theme" value="dark"/>
            <qualifier name="AlternateForm" value=""/>
            <qualifier name="DXFeatureLevel" value="DX9"/>
            <qualifier name="Configuration" value=""/>
            <qualifier name="DeviceFamily" value="Universal"/>
            <qualifier name="Custom" value=""/>
        </default>
        <indexer-config type="folder" foldernameAsQualifier="true" filenameAsQualifier="true" qualifierDelimiter="$"/>
        <indexer-config type="resw" convertDotsToSlashes="true" initialPath=""/>
        <indexer-config type="resjson" initialPath=""/>
        <indexer-config type="PRI"/>
    </index>
</resources>"""


SCCD_FILENAME = "PC/classicAppCompat.sccd"

SPECIAL_LOOKUP = object()

REGISTRY = {
    "HKCU\\Software\\Python\\PythonCore": {
        VER_DOT: {
            "DisplayName": APPX_DATA["DisplayName"],
            "SupportUrl": "https://www.python.org/",
            "SysArchitecture": SPECIAL_LOOKUP,
            "SysVersion": VER_DOT,
            "Version": "{}.{}.{}".format(VER_MAJOR, VER_MINOR, VER_MICRO),
            "InstallPath": {
                "": "[{AppVPackageRoot}]",
                "ExecutablePath": "[{{AppVPackageRoot}}]\\python{}.exe".format(VER_DOT),
                "WindowedExecutablePath": "[{{AppVPackageRoot}}]\\pythonw{}.exe".format(
                    VER_DOT
                ),
            },
            "Help": {
                "Main Python Documentation": {
                    "_condition": lambda ns: ns.include_chm,
                    "": "[{{AppVPackageRoot}}]\\Doc\\{}".format(PYTHON_CHM_NAME),
                },
                "Local Python Documentation": {
                    "_condition": lambda ns: ns.include_html_doc,
                    "": "[{AppVPackageRoot}]\\Doc\\html\\index.html",
                },
                "Online Python Documentation": {
                    "": "https://docs.python.org/{}".format(VER_DOT)
                },
            },
            "Idle": {
                "_condition": lambda ns: ns.include_idle,
                "": "[{AppVPackageRoot}]\\Lib\\idlelib\\idle.pyw",
            },
        }
    }
}


def get_packagefamilyname(name, publisher_id):
    class PACKAGE_ID(ctypes.Structure):
        _fields_ = [
            ("reserved", ctypes.c_uint32),
            ("processorArchitecture", ctypes.c_uint32),
            ("version", ctypes.c_uint64),
            ("name", ctypes.c_wchar_p),
            ("publisher", ctypes.c_wchar_p),
            ("resourceId", ctypes.c_wchar_p),
            ("publisherId", ctypes.c_wchar_p),
        ]
        _pack_ = 4

    pid = PACKAGE_ID(0, 0, 0, name, publisher_id, None, None)
    result = ctypes.create_unicode_buffer(256)
    result_len = ctypes.c_uint32(256)
    r = ctypes.windll.kernel32.PackageFamilyNameFromId(
        ctypes.byref(pid), ctypes.byref(result_len), result
    )
    if r:
        raise OSError(r, "failed to get package family name")
    return result.value[: result_len.value]


def _fixup_sccd(ns, sccd, new_hash=None):
    if not new_hash:
        return sccd

    NS = dict(s="http://schemas.microsoft.com/appx/2016/sccd")
    with open(sccd, "rb") as f:
        xml = ET.parse(f)

    pfn = get_packagefamilyname(APPX_DATA["Name"], APPX_DATA["Publisher"])

    ae = xml.find("s:AuthorizedEntities", NS)
    ae.clear()

    e = ET.SubElement(ae, ET.QName(NS["s"], "AuthorizedEntity"))
    e.set("AppPackageFamilyName", pfn)
    e.set("CertificateSignatureHash", new_hash)

    for e in xml.findall("s:Catalog", NS):
        e.text = "FFFF"

    sccd = ns.temp / sccd.name
    sccd.parent.mkdir(parents=True, exist_ok=True)
    with open(sccd, "wb") as f:
        xml.write(f, encoding="utf-8")

    return sccd


def find_or_add(xml, element, attr=None, always_add=False):
    if always_add:
        e = None
    else:
        q = element
        if attr:
            q += "[@{}='{}']".format(*attr)
        e = xml.find(q, APPXMANIFEST_NS)
    if e is None:
        prefix, _, name = element.partition(":")
        name = ET.QName(APPXMANIFEST_NS[prefix or ""], name)
        e = ET.SubElement(xml, name)
        if attr:
            e.set(*attr)
    return e


def _get_app(xml, appid):
    if appid:
        app = xml.find(
            "m:Applications/m:Application[@Id='{}']".format(appid), APPXMANIFEST_NS
        )
        if app is None:
            raise LookupError(appid)
    else:
        app = xml
    return app


def add_visual(xml, appid, data):
    app = _get_app(xml, appid)
    e = find_or_add(app, "uap:VisualElements")
    for i in data.items():
        e.set(*i)
    return e


def add_alias(xml, appid, alias, subsystem="windows"):
    app = _get_app(xml, appid)
    e = find_or_add(app, "m:Extensions")
    e = find_or_add(e, "uap5:Extension", ("Category", "windows.appExecutionAlias"))
    e = find_or_add(e, "uap5:AppExecutionAlias")
    e.set(ET.QName(APPXMANIFEST_NS["desktop4"], "Subsystem"), subsystem)
    e = find_or_add(e, "uap5:ExecutionAlias", ("Alias", alias))


def add_file_type(xml, appid, name, suffix, parameters='"%1"', info=None, logo=None):
    app = _get_app(xml, appid)
    e = find_or_add(app, "m:Extensions")
    e = find_or_add(e, "uap3:Extension", ("Category", "windows.fileTypeAssociation"))
    e = find_or_add(e, "uap3:FileTypeAssociation", ("Name", name))
    e.set("Parameters", parameters)
    if info:
        find_or_add(e, "uap:DisplayName").text = info
    if logo:
        find_or_add(e, "uap:Logo").text = logo
    e = find_or_add(e, "uap:SupportedFileTypes")
    if isinstance(suffix, str):
        suffix = [suffix]
    for s in suffix:
        ET.SubElement(e, ET.QName(APPXMANIFEST_NS["uap"], "FileType")).text = s


def add_application(
    ns, xml, appid, executable, aliases, visual_element, subsystem, file_types
):
    node = xml.find("m:Applications", APPXMANIFEST_NS)
    suffix = "_d.exe" if ns.debug else ".exe"
    app = ET.SubElement(
        node,
        ET.QName(APPXMANIFEST_NS[""], "Application"),
        {
            "Id": appid,
            "Executable": executable + suffix,
            "EntryPoint": "Windows.FullTrustApplication",
            ET.QName(APPXMANIFEST_NS["desktop4"], "SupportsMultipleInstances"): "true",
        },
    )
    if visual_element:
        add_visual(app, None, visual_element)
    for alias in aliases:
        add_alias(app, None, alias + suffix, subsystem)
    if file_types:
        add_file_type(app, None, *file_types)
    return app


def _get_registry_entries(ns, root="", d=None):
    r = root if root else PureWindowsPath("")
    if d is None:
        d = REGISTRY
    for key, value in d.items():
        if key == "_condition":
            continue
        if value is SPECIAL_LOOKUP:
            if key == "SysArchitecture":
                value = {
                    "win32": "32bit",
                    "amd64": "64bit",
                    "arm32": "32bit",
                    "arm64": "64bit",
                }[ns.arch]
            else:
                raise ValueError(f"Key '{key}' unhandled for special lookup")
        if isinstance(value, dict):
            cond = value.get("_condition")
            if cond and not cond(ns):
                continue
            fullkey = r
            for part in PureWindowsPath(key).parts:
                fullkey /= part
                if len(fullkey.parts) > 1:
                    yield str(fullkey), None, None
            yield from _get_registry_entries(ns, fullkey, value)
        elif len(r.parts) > 1:
            yield str(r), key, value


def add_registry_entries(ns, xml):
    e = find_or_add(xml, "m:Extensions")
    e = find_or_add(e, "rescap4:Extension")
    e.set("Category", "windows.classicAppCompatKeys")
    e.set("EntryPoint", "Windows.FullTrustApplication")
    e = ET.SubElement(e, ET.QName(APPXMANIFEST_NS["rescap4"], "ClassicAppCompatKeys"))
    for name, valuename, value in _get_registry_entries(ns):
        k = ET.SubElement(
            e, ET.QName(APPXMANIFEST_NS["rescap4"], "ClassicAppCompatKey")
        )
        k.set("Name", name)
        if value:
            k.set("ValueName", valuename)
            k.set("Value", value)
            k.set("ValueType", "REG_SZ")


def disable_registry_virtualization(xml):
    e = find_or_add(xml, "m:Properties")
    e = find_or_add(e, "desktop6:RegistryWriteVirtualization")
    e.text = "disabled"
    e = find_or_add(xml, "m:Capabilities")
    e = find_or_add(e, "rescap:Capability", ("Name", "unvirtualizedResources"))


def get_appxmanifest(ns):
    for k, v in APPXMANIFEST_NS.items():
        ET.register_namespace(k, v)
    ET.register_namespace("", APPXMANIFEST_NS["m"])

    xml = ET.parse(io.StringIO(APPXMANIFEST_TEMPLATE))
    NS = APPXMANIFEST_NS
    QN = ET.QName

    data = dict(APPX_DATA)
    for k, v in zip(APPX_PLATFORM_DATA["_keys"], APPX_PLATFORM_DATA[ns.arch]):
        data[k] = v

    node = xml.find("m:Identity", NS)
    for k in node.keys():
        value = data.get(k)
        if value:
            node.set(k, value)

    for node in xml.find("m:Properties", NS):
        value = data.get(node.tag.rpartition("}")[2])
        if value:
            node.text = value

    try:
        winver = tuple(int(i) for i in os.getenv("APPX_DATA_WINVER", "").split(".", maxsplit=3))
    except (TypeError, ValueError):
        winver = ()

    # Default "known good" version is 10.0.22000, first Windows 11 release
    winver = winver or (10, 0, 22000)

    if winver < (10, 0, 17763):
        winver = 10, 0, 17763
    find_or_add(xml, "m:Dependencies/m:TargetDeviceFamily").set(
        "MaxVersionTested", "{}.{}.{}.{}".format(*(winver + (0, 0, 0, 0)[:4]))
    )

    # Only for Python 3.11 and later. Older versions do not disable virtualization
    if (VER_MAJOR, VER_MINOR) >= (3, 11) and winver > (10, 0, 17763):
        disable_registry_virtualization(xml)

    app = add_application(
        ns,
        xml,
        "Python",
        "python{}".format(VER_DOT),
        ["python", "python{}".format(VER_MAJOR), "python{}".format(VER_DOT)],
        PYTHON_VE_DATA,
        "console",
        ("python.file", [".py"], '"%1" %*', "Python File", PY_PNG),
    )

    add_application(
        ns,
        xml,
        "PythonW",
        "pythonw{}".format(VER_DOT),
        ["pythonw", "pythonw{}".format(VER_MAJOR), "pythonw{}".format(VER_DOT)],
        PYTHONW_VE_DATA,
        "windows",
        ("python.windowedfile", [".pyw"], '"%1" %*', "Python File (no console)", PY_PNG),
    )

    if ns.include_pip and ns.include_launchers:
        add_application(
            ns,
            xml,
            "Pip",
            "pip{}".format(VER_DOT),
            ["pip", "pip{}".format(VER_MAJOR), "pip{}".format(VER_DOT)],
            PIP_VE_DATA,
            "console",
            ("python.wheel", [".whl"], 'install "%1"', "Python Wheel"),
        )

    if ns.include_idle and ns.include_launchers:
        add_application(
            ns,
            xml,
            "Idle",
            "idle{}".format(VER_DOT),
            ["idle", "idle{}".format(VER_MAJOR), "idle{}".format(VER_DOT)],
            IDLE_VE_DATA,
            "windows",
            None,
        )

    if (ns.source / SCCD_FILENAME).is_file():
        add_registry_entries(ns, xml)
        node = xml.find("m:Capabilities", NS)
        node = ET.SubElement(node, QN(NS["uap4"], "CustomCapability"))
        node.set("Name", "Microsoft.classicAppCompat_8wekyb3d8bbwe")

    buffer = io.BytesIO()
    xml.write(buffer, encoding="utf-8", xml_declaration=True)
    return buffer.getbuffer()


def get_resources_xml(ns):
    return RESOURCES_XML_TEMPLATE.encode("utf-8")


def get_appx_layout(ns):
    if not ns.include_appxmanifest:
        return

    yield "AppxManifest.xml", ("AppxManifest.xml", get_appxmanifest(ns))
    yield "_resources.xml", ("_resources.xml", get_resources_xml(ns))
    icons = ns.source / "PC" / "icons"
    for px in [44, 50, 150]:
        src = icons / "pythonx{}.png".format(px)
        yield f"_resources/pythonx{px}.png", src
        yield f"_resources/pythonx{px}$targetsize-{px}_altform-unplated.png", src
    for px in [44, 150]:
        src = icons / "pythonwx{}.png".format(px)
        yield f"_resources/pythonwx{px}.png", src
        yield f"_resources/pythonwx{px}$targetsize-{px}_altform-unplated.png", src
    if ns.include_idle and ns.include_launchers:
        for px in [44, 150]:
            src = icons / "idlex{}.png".format(px)
            yield f"_resources/idlex{px}.png", src
            yield f"_resources/idlex{px}$targetsize-{px}_altform-unplated.png", src
    yield f"_resources/py.png", icons / "py.png"
    sccd = ns.source / SCCD_FILENAME
    if sccd.is_file():
        # This should only be set for side-loading purposes.
        sccd = _fixup_sccd(ns, sccd, os.getenv("APPX_DATA_SHA256"))
        yield sccd.name, sccd

### File: arch.py

In [ ]:
from struct import unpack
from .constants import *
from .logging import *

def calculate_from_build_dir(root):
    candidates = [
        root / PYTHON_DLL_NAME,
        root / FREETHREADED_PYTHON_DLL_NAME,
        *root.glob("*.dll"),
        *root.glob("*.pyd"),
        # Check EXE last because it's easier to have cross-platform EXE
        *root.glob("*.exe"),
    ]

    ARCHS = {
        b"PE\0\0\x4c\x01": "win32",
        b"PE\0\0\x64\x86": "amd64",
        b"PE\0\0\x64\xAA": "arm64"
    }

    first_exc = None
    for pe in candidates:
        try:
            # Read the PE header to grab the machine type
            with open(pe, "rb") as f:
                f.seek(0x3C)
                offset = int.from_bytes(f.read(4), "little")
                f.seek(offset)
                arch = ARCHS[f.read(6)]
        except (FileNotFoundError, PermissionError, LookupError) as ex:
            log_debug("Failed to open {}: {}", pe, ex)
            continue
        log_info("Inferred architecture {} from {}", arch, pe)
        return arch

### File: catalog.py

In [ ]:
"""
File generation for catalog signing non-binary contents.
"""

__author__ = "Steve Dower <steve.dower@python.org>"
__version__ = "3.8"


__all__ = ["PYTHON_CAT_NAME", "PYTHON_CDF_NAME"]


def public(f):
    __all__.append(f.__name__)
    return f


PYTHON_CAT_NAME = "python.cat"
PYTHON_CDF_NAME = "python.cdf"


CATALOG_TEMPLATE = r"""[CatalogHeader]
Name={target.stem}.cat
ResultDir={target.parent}
PublicVersion=1
CatalogVersion=2
HashAlgorithms=SHA256
PageHashes=false
EncodingType=

[CatalogFiles]
"""


def can_sign(file):
    return file.is_file() and file.stat().st_size


@public
def write_catalog(target, files):
    with target.open("w", encoding="utf-8") as cat:
        cat.write(CATALOG_TEMPLATE.format(target=target))
        cat.writelines("<HASH>{}={}\n".format(n, f) for n, f in files if can_sign(f))

### File: constants.py

In [ ]:
"""
Constants for generating the layout.
"""

__author__ = "Steve Dower <steve.dower@python.org>"
__version__ = "3.8"

import os
import pathlib
import re
import struct
import sys


def _unpack_hexversion():
    try:
        hexversion = int(os.getenv("PYTHON_HEXVERSION"), 16)
        return struct.pack(">i", hexversion)
    except (TypeError, ValueError):
        pass
    if os.getenv("PYTHONINCLUDE"):
        try:
            return _read_patchlevel_version(pathlib.Path(os.getenv("PYTHONINCLUDE")))
        except OSError:
            pass
    return struct.pack(">i", sys.hexversion)


def _get_suffix(field4):
    name = {0xA0: "a", 0xB0: "b", 0xC0: "rc"}.get(field4 & 0xF0, "")
    if name:
        serial = field4 & 0x0F
        return f"{name}{serial}"
    return ""


def _read_patchlevel_version(sources):
    if not sources.match("Include"):
        sources /= "Include"
    values = {}
    with open(sources / "patchlevel.h", "r", encoding="utf-8") as f:
        for line in f:
            m = re.match(r'#\s*define\s+(PY_\S+?)\s+(\S+)', line.strip(), re.I)
            if m and m.group(2):
                v = m.group(2)
                if v.startswith('"'):
                    v = v[1:-1]
                else:
                    v = values.get(v, v)
                    if isinstance(v, str):
                        try:
                            v = int(v, 16 if v.startswith("0x") else 10)
                        except ValueError:
                            pass
                values[m.group(1)] = v
    return (
        values["PY_MAJOR_VERSION"],
        values["PY_MINOR_VERSION"],
        values["PY_MICRO_VERSION"],
        values["PY_RELEASE_LEVEL"] << 4 | values["PY_RELEASE_SERIAL"],
    )


def check_patchlevel_version(sources):
    got = _read_patchlevel_version(sources)
    if got != (VER_MAJOR, VER_MINOR, VER_MICRO, VER_FIELD4):
        return f"{got[0]}.{got[1]}.{got[2]}{_get_suffix(got[3])}"


VER_MAJOR, VER_MINOR, VER_MICRO, VER_FIELD4 = _unpack_hexversion()
VER_SUFFIX = _get_suffix(VER_FIELD4)
VER_FIELD3 = VER_MICRO << 8 | VER_FIELD4
VER_DOT = "{}.{}".format(VER_MAJOR, VER_MINOR)

PYTHON_DLL_NAME = "python{}{}.dll".format(VER_MAJOR, VER_MINOR)
PYTHON_STABLE_DLL_NAME = "python{}.dll".format(VER_MAJOR)
PYTHON_ZIP_NAME = "python{}{}.zip".format(VER_MAJOR, VER_MINOR)
PYTHON_PTH_NAME = "python{}{}._pth".format(VER_MAJOR, VER_MINOR)

PYTHON_CHM_NAME = "python{}{}{}{}.chm".format(
    VER_MAJOR, VER_MINOR, VER_MICRO, VER_SUFFIX
)

FREETHREADED_PYTHON_DLL_NAME = "python{}{}t.dll".format(VER_MAJOR, VER_MINOR)
FREETHREADED_PYTHON_STABLE_DLL_NAME = "python{}t.dll".format(VER_MAJOR)

### File: filesets.py

In [ ]:
"""
File sets and globbing helper for make_layout.
"""

__author__ = "Steve Dower <steve.dower@python.org>"
__version__ = "3.8"

import os


class FileStemSet:
    def __init__(self, *patterns):
        self._names = set()
        self._prefixes = []
        self._suffixes = []
        for p in map(os.path.normcase, patterns):
            if p.endswith("*"):
                self._prefixes.append(p[:-1])
            elif p.startswith("*"):
                self._suffixes.append(p[1:])
            else:
                self._names.add(p)

    def _make_name(self, f):
        return os.path.normcase(f.stem)

    def __contains__(self, f):
        bn = self._make_name(f)
        return (
            bn in self._names
            or any(map(bn.startswith, self._prefixes))
            or any(map(bn.endswith, self._suffixes))
        )


class FileNameSet(FileStemSet):
    def _make_name(self, f):
        return os.path.normcase(f.name)


class FileSuffixSet:
    def __init__(self, *patterns):
        self._names = set()
        self._prefixes = []
        self._suffixes = []
        for p in map(os.path.normcase, patterns):
            if p.startswith("*."):
                self._names.add(p[1:])
            elif p.startswith("*"):
                self._suffixes.append(p[1:])
            elif p.endswith("*"):
                self._prefixes.append(p[:-1])
            elif p.startswith("."):
                self._names.add(p)
            else:
                self._names.add("." + p)

    def _make_name(self, f):
        return os.path.normcase(f.suffix)

    def __contains__(self, f):
        bn = self._make_name(f)
        return (
            bn in self._names
            or any(map(bn.startswith, self._prefixes))
            or any(map(bn.endswith, self._suffixes))
        )


def _rglob(root, pattern, condition):
    dirs = [root]
    recurse = pattern[:3] in {"**/", "**\\"}
    if recurse:
        pattern = pattern[3:]

    while dirs:
        d = dirs.pop(0)
        if recurse:
            dirs.extend(
                filter(
                    condition, (type(root)(f2) for f2 in os.scandir(d) if f2.is_dir())
                )
            )
        yield from (
            (f.relative_to(root), f)
            for f in d.glob(pattern)
            if f.is_file() and condition(f)
        )


def _return_true(f):
    return True


def rglob(root, patterns, condition=None):
    if not os.path.isdir(root):
        return
    if isinstance(patterns, tuple):
        for p in patterns:
            yield from _rglob(root, p, condition or _return_true)
    else:
        yield from _rglob(root, patterns, condition or _return_true)

### File: logging.py

In [ ]:
"""
Logging support for make_layout.
"""

__author__ = "Steve Dower <steve.dower@python.org>"
__version__ = "3.8"

import logging
import sys

__all__ = []

LOG = None
HAS_ERROR = False


def public(f):
    __all__.append(f.__name__)
    return f


@public
def configure_logger(ns):
    global LOG
    if LOG:
        return

    LOG = logging.getLogger("make_layout")
    LOG.level = logging.DEBUG

    if ns.v:
        s_level = max(logging.ERROR - ns.v * 10, logging.DEBUG)
        f_level = max(logging.WARNING - ns.v * 10, logging.DEBUG)
    else:
        s_level = logging.ERROR
        f_level = logging.INFO

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("{levelname:8s} {message}", style="{"))
    handler.setLevel(s_level)
    LOG.addHandler(handler)

    if ns.log:
        handler = logging.FileHandler(ns.log, encoding="utf-8", delay=True)
        handler.setFormatter(
            logging.Formatter("[{asctime}]{levelname:8s}: {message}", style="{")
        )
        handler.setLevel(f_level)
        LOG.addHandler(handler)


class BraceMessage:
    def __init__(self, fmt, *args, **kwargs):
        self.fmt = fmt
        self.args = args
        self.kwargs = kwargs

    def __str__(self):
        return self.fmt.format(*self.args, **self.kwargs)


@public
def log_debug(msg, *args, **kwargs):
    return LOG.debug(BraceMessage(msg, *args, **kwargs))


@public
def log_info(msg, *args, **kwargs):
    return LOG.info(BraceMessage(msg, *args, **kwargs))


@public
def log_warning(msg, *args, **kwargs):
    return LOG.warning(BraceMessage(msg, *args, **kwargs))


@public
def log_error(msg, *args, **kwargs):
    global HAS_ERROR
    HAS_ERROR = True
    return LOG.error(BraceMessage(msg, *args, **kwargs))


@public
def log_exception(msg, *args, **kwargs):
    global HAS_ERROR
    HAS_ERROR = True
    return LOG.exception(BraceMessage(msg, *args, **kwargs))


@public
def error_was_logged():
    return HAS_ERROR

### File: nuspec.py

In [ ]:
"""
Provides .props file.
"""

import os
import sys

from .constants import *

__all__ = ["get_nuspec_layout"]

PYTHON_NUSPEC_NAME = "python.nuspec"

NUSPEC_DATA = {
    "PYTHON_TAG": VER_DOT,
    "PYTHON_VERSION": os.getenv("PYTHON_NUSPEC_VERSION"),
    "FILELIST": r'    <file src="**\*" exclude="python.png" target="tools" />',
    "GIT": sys._git,
}

NUSPEC_PLATFORM_DATA = dict(
    _keys=("PYTHON_BITNESS", "PACKAGENAME", "PACKAGETITLE"),
    win32=("32-bit", "pythonx86", "Python (32-bit)"),
    amd64=("64-bit", "python", "Python"),
    arm32=("ARM", "pythonarm", "Python (ARM)"),
    arm64=("ARM64", "pythonarm64", "Python (ARM64)"),
    win32t=("32-bit free-threaded", "pythonx86-freethreaded", "Python (32-bit, free-threaded)"),
    amd64t=("64-bit free-threaded", "python-freethreaded", "Python (free-threaded)"),
    arm32t=("ARM free-threaded", "pythonarm-freethreaded", "Python (ARM, free-threaded)"),
    arm64t=("ARM64 free-threaded", "pythonarm64-freethreaded", "Python (ARM64, free-threaded)"),
)

if not NUSPEC_DATA["PYTHON_VERSION"]:
    NUSPEC_DATA["PYTHON_VERSION"] = "{}.{}{}{}".format(
        VER_DOT, VER_MICRO, "-" if VER_SUFFIX else "", VER_SUFFIX
    )

FILELIST_WITH_PROPS = r"""    <file src="**\*" exclude="python.png;python.props" target="tools" />
    <file src="python.props" target="build\native" />"""

NUSPEC_TEMPLATE = r"""<?xml version="1.0"?>
<package>
  <metadata>
    <id>{PACKAGENAME}</id>
    <title>{PACKAGETITLE}</title>
    <version>{PYTHON_VERSION}</version>
    <authors>Python Software Foundation</authors>
    <license type="file">tools\LICENSE.txt</license>
    <projectUrl>https://www.python.org/</projectUrl>
    <description>Installs {PYTHON_BITNESS} Python for use in build scenarios.</description>
    <icon>images\python.png</icon>
    <iconUrl>https://www.python.org/static/favicon.ico</iconUrl>
    <tags>python</tags>
    <repository type="git" url="https://github.com/Python/CPython.git" commit="{GIT[2]}" />
  </metadata>
  <files>
    <file src="python.png" target="images" />
{FILELIST}
  </files>
</package>
"""


def _get_nuspec_data_overrides(ns):
    arch = ns.arch
    if ns.include_freethreaded:
        arch += "t"
    for k, v in zip(NUSPEC_PLATFORM_DATA["_keys"], NUSPEC_PLATFORM_DATA[arch]):
        ev = os.getenv("PYTHON_NUSPEC_" + k)
        if ev:
            yield k, ev
        yield k, v


def get_nuspec_layout(ns):
    if ns.include_all or ns.include_nuspec:
        data = dict(NUSPEC_DATA)
        for k, v in _get_nuspec_data_overrides(ns):
            if not data.get(k):
                data[k] = v
        if ns.include_all or ns.include_props:
            data["FILELIST"] = FILELIST_WITH_PROPS
        nuspec = NUSPEC_TEMPLATE.format_map(data)
        yield "python.nuspec", ("python.nuspec", nuspec.encode("utf-8"))
        yield "python.png", ns.source / "PC" / "icons" / "logox128.png"

### File: options.py

In [ ]:
"""
List of optional components.
"""

__author__ = "Steve Dower <steve.dower@python.org>"
__version__ = "3.8"


__all__ = []


def public(f):
    __all__.append(f.__name__)
    return f


OPTIONS = {
    "stable": {"help": "stable ABI stub"},
    "pip": {"help": "pip"},
    "pip-user": {"help": "pip.ini file for default --user"},
    "tcltk": {"help": "Tcl, Tk and tkinter"},
    "idle": {"help": "Idle"},
    "tests": {"help": "test suite"},
    "tools": {"help": "tools"},
    "venv": {"help": "venv"},
    "dev": {"help": "headers and libs"},
    "symbols": {"help": "symbols"},
    "underpth": {"help": "a python._pth file", "not-in-all": True},
    "launchers": {"help": "specific launchers"},
    "appxmanifest": {"help": "an appxmanifest"},
    "props": {"help": "a python.props file"},
    "nuspec": {"help": "a python.nuspec file"},
    "chm": {"help": "the CHM documentation"},
    "html-doc": {"help": "the HTML documentation"},
    "freethreaded": {"help": "freethreaded binaries", "not-in-all": True},
    "alias": {"help": "aliased python.exe entry-point binaries"},
    "alias3": {"help": "aliased python3.exe entry-point binaries"},
    "alias3x": {"help": "aliased python3.x.exe entry-point binaries"},
    "install-json": {"help": "a PyManager __install__.json file"},
    "install-embed-json": {"help": "a PyManager __install__.json file for embeddable distro"},
    "install-test-json": {"help": "a PyManager __install__.json for the test distro"},
}


PRESETS = {
    "appx": {
        "help": "APPX package",
        "options": [
            "stable",
            "pip",
            "tcltk",
            "idle",
            "venv",
            "dev",
            "launchers",
            "appxmanifest",
            "alias",
            "alias3x",
            # XXX: Disabled for now "precompile",
        ],
    },
    "nuget": {
        "help": "nuget package",
        "options": [
            "dev",
            "pip",
            "stable",
            "venv",
            "props",
            "nuspec",
            "alias",
        ],
    },
    "iot": {"help": "Windows IoT Core", "options": ["alias", "stable", "pip"]},
    "default": {
        "help": "development kit package",
        "options": [
            "stable",
            "pip",
            "tcltk",
            "idle",
            "tests",
            "venv",
            "dev",
            "symbols",
            "html-doc",
            "alias",
        ],
    },
    "embed": {
        "help": "embeddable package",
        "options": [
            "alias",
            "stable",
            "zip-lib",
            "flat-dlls",
            "underpth",
            "precompile",
        ],
    },
    "pymanager": {
        "help": "PyManager package",
        "options": [
            "stable",
            "pip",
            "tcltk",
            "idle",
            "venv",
            "dev",
            "html-doc",
            "install-json",
        ],
    },
    "pymanager-test": {
        "help": "PyManager test package",
        "options": [
            "stable",
            "pip",
            "tcltk",
            "idle",
            "venv",
            "dev",
            "html-doc",
            "symbols",
            "tests",
            "install-test-json",
        ],
    },
}


@public
def get_argparse_options():
    for opt, info in OPTIONS.items():
        help = "When specified, includes {}".format(info["help"])
        if info.get("not-in-all"):
            help = "{}. Not affected by --include-all".format(help)

        yield "--include-{}".format(opt), help

    for opt, info in PRESETS.items():
        help = "When specified, includes default options for {}".format(info["help"])
        yield "--preset-{}".format(opt), help


def ns_get(ns, key, default=False):
    return getattr(ns, key.replace("-", "_"), default)


def ns_set(ns, key, value=True):
    k1 = key.replace("-", "_")
    k2 = "include_{}".format(k1)
    if hasattr(ns, k2):
        setattr(ns, k2, value)
    elif hasattr(ns, k1):
        setattr(ns, k1, value)
    else:
        raise AttributeError("no argument named '{}'".format(k1))


@public
def update_presets(ns):
    for preset, info in PRESETS.items():
        if ns_get(ns, "preset-{}".format(preset)):
            for opt in info["options"]:
                ns_set(ns, opt)

    if ns.include_all:
        for opt in OPTIONS:
            if OPTIONS[opt].get("not-in-all"):
                continue
            ns_set(ns, opt)

### File: pip.py

In [ ]:
"""
Extraction and file list generation for pip.
"""

__author__ = "Steve Dower <steve.dower@python.org>"
__version__ = "3.8"


import os
import shutil
import subprocess
import sys

from .filesets import *

__all__ = ["extract_pip_files", "get_pip_layout"]


def get_pip_dir(ns):
    if ns.copy:
        if ns.zip_lib:
            return ns.copy / "packages"
        return ns.copy / "Lib" / "site-packages"
    else:
        return ns.temp / "packages"


def get_pip_layout(ns):
    pip_dir = get_pip_dir(ns)
    if not pip_dir.is_dir():
        log_warning("Failed to find {} - pip will not be included", pip_dir)
    else:
        pkg_root = "packages/{}" if ns.zip_lib else "Lib/site-packages/{}"
        for dest, src in rglob(pip_dir, "**/*"):
            yield pkg_root.format(dest), src
        if ns.include_pip_user:
            content = "\n".join(
                "[{}]\nuser=yes".format(n)
                for n in ["install", "uninstall", "freeze", "list"]
            )
            yield "pip.ini", ("pip.ini", content.encode())


def extract_pip_files(ns):
    dest = get_pip_dir(ns)
    try:
        dest.mkdir(parents=True, exist_ok=False)
    except IOError:
        return

    src = ns.source / "Lib" / "ensurepip" / "_bundled"

    ns.temp.mkdir(parents=True, exist_ok=True)
    wheels = [shutil.copy(whl, ns.temp) for whl in src.glob("*.whl")]
    search_path = os.pathsep.join(wheels)
    if os.environ.get("PYTHONPATH"):
        search_path += ";" + os.environ["PYTHONPATH"]

    env = os.environ.copy()
    env["PYTHONPATH"] = search_path

    output = subprocess.check_output(
        [
            sys.executable,
            "-m",
            "pip",
            "--no-color",
            "install",
            "pip",
            "--upgrade",
            "--target",
            str(dest),
            "--no-index",
            "--no-compile",
            "--no-cache-dir",
            "-f",
            str(src),
            "--only-binary",
            ":all:",
        ],
        env=env,
    )

    try:
        shutil.rmtree(dest / "bin")
    except OSError:
        pass

    for file in wheels:
        try:
            os.remove(file)
        except OSError:
            pass

### File: props.py

In [ ]:
"""
Provides .props file.
"""

import os

from .constants import *

__all__ = ["get_props_layout"]

PYTHON_PROPS_NAME = "python.props"

PROPS_DATA = {
    "PYTHON_TAG": VER_DOT,
    "PYTHON_VERSION": os.getenv("PYTHON_NUSPEC_VERSION"),
    "PYTHON_PLATFORM": os.getenv("PYTHON_PROPS_PLATFORM"),
    "PYTHON_TARGET": "",
}

if not PROPS_DATA["PYTHON_VERSION"]:
    PROPS_DATA["PYTHON_VERSION"] = "{}.{}{}{}".format(
        VER_DOT, VER_MICRO, "-" if VER_SUFFIX else "", VER_SUFFIX
    )

PROPS_DATA["PYTHON_TARGET"] = "_GetPythonRuntimeFilesDependsOn{}{}_{}".format(
    VER_MAJOR, VER_MINOR, PROPS_DATA["PYTHON_PLATFORM"]
)

PROPS_TEMPLATE = r"""<?xml version="1.0" encoding="utf-8"?>
<Project xmlns="http://schemas.microsoft.com/developer/msbuild/2003">
  <PropertyGroup Condition="$(Platform) == '{PYTHON_PLATFORM}'">
    <PythonHome Condition="$(PythonHome) == ''">$([System.IO.Path]::GetFullPath("$(MSBuildThisFileDirectory)\..\..\tools"))</PythonHome>
    <PythonInclude>$(PythonHome)\include</PythonInclude>
    <PythonLibs>$(PythonHome)\libs</PythonLibs>
    <PythonTag>{PYTHON_TAG}</PythonTag>
    <PythonVersion>{PYTHON_VERSION}</PythonVersion>

    <IncludePythonExe Condition="$(IncludePythonExe) == ''">true</IncludePythonExe>
    <IncludeVEnv Condition="$(IncludeVEnv) == ''">false</IncludeVEnv>

    <GetPythonRuntimeFilesDependsOn>{PYTHON_TARGET};$(GetPythonRuntimeFilesDependsOn)</GetPythonRuntimeFilesDependsOn>
  </PropertyGroup>

  <ItemDefinitionGroup Condition="$(Platform) == '{PYTHON_PLATFORM}'">
    <ClCompile>
      <AdditionalIncludeDirectories>$(PythonInclude);%(AdditionalIncludeDirectories)</AdditionalIncludeDirectories>
      <RuntimeLibrary>MultiThreadedDLL</RuntimeLibrary>
    </ClCompile>
    <Link>
      <AdditionalLibraryDirectories>$(PythonLibs);%(AdditionalLibraryDirectories)</AdditionalLibraryDirectories>
    </Link>
  </ItemDefinitionGroup>

  <Target Name="GetPythonRuntimeFiles" Returns="@(PythonRuntime)" DependsOnTargets="$(GetPythonRuntimeFilesDependsOn)" />

  <Target Name="{PYTHON_TARGET}" Returns="@(PythonRuntime)">
    <ItemGroup>
      <_PythonRuntimeExe Include="$(PythonHome)\python*.dll" />
      <_PythonRuntimeExe Include="$(PythonHome)\python*.exe" Condition="$(IncludePythonExe) == 'true'" />
      <_PythonRuntimeExe>
        <Link>%(Filename)%(Extension)</Link>
      </_PythonRuntimeExe>
      <_PythonRuntimeDlls Include="$(PythonHome)\DLLs\*.pyd" />
      <_PythonRuntimeDlls Include="$(PythonHome)\DLLs\*.dll" />
      <_PythonRuntimeDlls>
        <Link>DLLs\%(Filename)%(Extension)</Link>
      </_PythonRuntimeDlls>
      <_PythonRuntimeLib Include="$(PythonHome)\Lib\**\*" Exclude="$(PythonHome)\Lib\**\*.pyc;$(PythonHome)\Lib\site-packages\**\*" />
      <_PythonRuntimeLib Remove="$(PythonHome)\Lib\ensurepip\**\*" Condition="$(IncludeVEnv) != 'true'" />
      <_PythonRuntimeLib Remove="$(PythonHome)\Lib\venv\**\*" Condition="$(IncludeVEnv) != 'true'" />
      <_PythonRuntimeLib>
        <Link>Lib\%(RecursiveDir)%(Filename)%(Extension)</Link>
      </_PythonRuntimeLib>
      <PythonRuntime Include="@(_PythonRuntimeExe);@(_PythonRuntimeDlls);@(_PythonRuntimeLib)" />
    </ItemGroup>

    <Message Importance="low" Text="Collected Python runtime from $(PythonHome):%0D%0A@(PythonRuntime->'  %(Link)','%0D%0A')" />
  </Target>
</Project>
"""


def get_props_layout(ns):
    if ns.include_all or ns.include_props:
        # TODO: Filter contents of props file according to included/excluded items
        d = dict(PROPS_DATA)
        if not d.get("PYTHON_PLATFORM"):
            d["PYTHON_PLATFORM"] = {
                "win32": "Win32",
                "amd64": "X64",
                "arm32": "ARM",
                "arm64": "ARM64",
            }[ns.arch]
        props = PROPS_TEMPLATE.format_map(d)
        yield "python.props", ("python.props", props.encode("utf-8"))

### File: pymanager.py

In [ ]:
from .constants import *

URL_BASE = "https://www.python.org/ftp/python/"

XYZ_VERSION = f"{VER_MAJOR}.{VER_MINOR}.{VER_MICRO}"
WIN32_VERSION = f"{VER_MAJOR}.{VER_MINOR}.{VER_MICRO}.{VER_FIELD4}"
FULL_VERSION = f"{VER_MAJOR}.{VER_MINOR}.{VER_MICRO}{VER_SUFFIX}"


def _not_empty(n, key=None):
    result = []
    for i in n:
        if key:
            i_l = i[key]
        else:
            i_l = i
        if not i_l:
            continue
        result.append(i)
    return result


def calculate_install_json(ns, *, for_embed=False, for_test=False):
    TARGET = "python.exe"
    TARGETW = "pythonw.exe"

    SYS_ARCH = {
        "win32": "32bit",
        "amd64": "64bit",
        "arm64": "64bit", # Unfortunate, but this is how it's spec'd
    }[ns.arch]
    TAG_ARCH = {
        "win32": "-32",
        "amd64": "-64",
        "arm64": "-arm64",
    }[ns.arch]

    COMPANY = "PythonCore"
    DISPLAY_NAME = "Python"
    TAG_SUFFIX = ""
    ALIAS_PREFIX = "python"
    ALIAS_WPREFIX = "pythonw"
    FILE_PREFIX = "python-"
    FILE_SUFFIX = f"-{ns.arch}"
    DISPLAY_TAGS = [{
        "win32": "32-bit",
        "amd64": "",
        "arm64": "ARM64",
    }[ns.arch]]

    if for_test:
        # Packages with the test suite come under a different Company
        COMPANY = "PythonTest"
        DISPLAY_TAGS.append("with tests")
        FILE_SUFFIX = f"-test-{ns.arch}"
    if for_embed:
        # Embeddable distro comes under a different Company
        COMPANY = "PythonEmbed"
        TARGETW = None
        ALIAS_PREFIX = None
        ALIAS_WPREFIX = None
        DISPLAY_TAGS.append("embeddable")
        # Deliberately name the file differently from the existing distro
        # so we can republish old versions without replacing files.
        FILE_SUFFIX = f"-embeddable-{ns.arch}"
    if ns.include_freethreaded:
        # Free-threaded distro comes with a tag suffix
        TAG_SUFFIX = "t"
        TARGET = f"python{VER_MAJOR}.{VER_MINOR}t.exe"
        TARGETW = f"pythonw{VER_MAJOR}.{VER_MINOR}t.exe"
        DISPLAY_TAGS.append("free-threaded")
        FILE_SUFFIX = f"t-{ns.arch}"

    FULL_TAG = f"{VER_MAJOR}.{VER_MINOR}.{VER_MICRO}{VER_SUFFIX}{TAG_SUFFIX}"
    FULL_ARCH_TAG = f"{FULL_TAG}{TAG_ARCH}"
    XY_TAG = f"{VER_MAJOR}.{VER_MINOR}{TAG_SUFFIX}"
    XY_ARCH_TAG = f"{XY_TAG}{TAG_ARCH}"
    X_TAG = f"{VER_MAJOR}{TAG_SUFFIX}"
    X_ARCH_TAG = f"{X_TAG}{TAG_ARCH}"

    # Tag used in runtime ID (for side-by-side install/updates)
    ID_TAG = XY_ARCH_TAG
    # Tag shown in 'py list' output.
    # gh-139810: We only include '-dev' here for prereleases, even though it
    # works for final releases too.
    DISPLAY_TAG = f"{XY_TAG}-dev{TAG_ARCH}" if VER_SUFFIX else XY_ARCH_TAG
    # Tag used for PEP 514 registration
    SYS_WINVER = XY_TAG + (TAG_ARCH if TAG_ARCH != '-64' else '')

    DISPLAY_SUFFIX = ", ".join(i for i in DISPLAY_TAGS if i)
    if DISPLAY_SUFFIX:
        DISPLAY_SUFFIX = f" ({DISPLAY_SUFFIX})"
    DISPLAY_VERSION = f"{XYZ_VERSION}{VER_SUFFIX}{DISPLAY_SUFFIX}"

    STD_RUN_FOR = []
    STD_ALIAS = []
    STD_PEP514 = []
    STD_START = []
    STD_UNINSTALL = []

    # The list of 'py install <TAG>' tags that will match this runtime.
    # Architecture should always be included here because PyManager will add it.
    INSTALL_TAGS = [
        FULL_ARCH_TAG,
        XY_ARCH_TAG,
        X_ARCH_TAG,
        # gh-139810: The -dev tags are always included so that the latest
        # release is chosen, no matter whether it's prerelease or final.
        f"{XY_TAG}-dev{TAG_ARCH}" if XY_TAG else "",
        f"{X_TAG}-dev{TAG_ARCH}" if X_TAG else "",
    ]

    # Generate run-for entries for each target.
    # Again, include architecture because PyManager will add it.
    for base in [
        {"target": TARGET},
        {"target": TARGETW, "windowed": 1},
    ]:
        if not base["target"]:
            continue
        STD_RUN_FOR.extend([
            {**base, "tag": FULL_ARCH_TAG},
            {**base, "tag": f"{XY_TAG}-dev{TAG_ARCH}" if XY_TAG else ""},
            {**base, "tag": f"{X_TAG}-dev{TAG_ARCH}" if X_TAG else ""},
        ])
        if XY_TAG:
            STD_RUN_FOR.append({**base, "tag": XY_ARCH_TAG})
        if X_TAG:
            STD_RUN_FOR.append({**base, "tag": X_ARCH_TAG})

    # Generate alias entries for each target. We need both arch and non-arch
    # versions as well as windowed/non-windowed versions to make sure that all
    # necessary aliases are created.
    for prefix, base in (
        (ALIAS_PREFIX, {"target": TARGET}),
        (ALIAS_WPREFIX, {"target": TARGETW, "windowed": 1}),
    ):
        if not prefix:
            continue
        if not base["target"]:
            continue
        if XY_TAG:
            STD_ALIAS.extend([
                {**base, "name": f"{prefix}{XY_TAG}.exe"},
                {**base, "name": f"{prefix}{XY_ARCH_TAG}.exe"},
            ])
        if X_TAG:
            STD_ALIAS.extend([
                {**base, "name": f"{prefix}{X_TAG}.exe"},
                {**base, "name": f"{prefix}{X_ARCH_TAG}.exe"},
            ])

    if SYS_WINVER:
        STD_PEP514.append({
            "kind": "pep514",
            "Key": rf"{COMPANY}\{SYS_WINVER}",
            "DisplayName": f"{DISPLAY_NAME} {DISPLAY_VERSION}",
            "SupportUrl": "https://www.python.org/",
            "SysArchitecture": SYS_ARCH,
            "SysVersion": VER_DOT,
            "Version": FULL_VERSION,
            "InstallPath": {
                "_": "%PREFIX%",
                "ExecutablePath": f"%PREFIX%{TARGET}",
                # WindowedExecutablePath is added below
            },
            "Help": {
                "Online Python Documentation": {
                    "_": f"https://docs.python.org/{VER_DOT}/"
                },
            },
        })

    STD_START.append({
        "kind": "start",
        "Name": f"{DISPLAY_NAME} {VER_DOT}{DISPLAY_SUFFIX}",
        "Items": [
            {
                "Name": f"{DISPLAY_NAME} {VER_DOT}{DISPLAY_SUFFIX}",
                "Target": f"%PREFIX%{TARGET}",
                "Icon": f"%PREFIX%{TARGET}",
            },
            {
                "Name": f"{DISPLAY_NAME} {VER_DOT} Online Documentation",
                "Icon": r"%SystemRoot%\System32\SHELL32.dll",
                "IconIndex": 13,
                "Target": f"https://docs.python.org/{VER_DOT}/",
            },
            # IDLE and local documentation items are added below
        ],
    })

    if TARGETW and STD_PEP514:
        STD_PEP514[0]["InstallPath"]["WindowedExecutablePath"] = f"%PREFIX%{TARGETW}"

    if ns.include_idle:
        STD_START[0]["Items"].append({
            "Name": f"IDLE (Python {VER_DOT}{DISPLAY_SUFFIX})",
            "Target": f"%PREFIX%{TARGETW or TARGET}",
            "Arguments": r'"%PREFIX%Lib\idlelib\idle.pyw"',
            "Icon": r"%PREFIX%Lib\idlelib\Icons\idle.ico",
            "IconIndex": 0,
        })
        STD_START[0]["Items"].append({
            "Name": f"PyDoc (Python {VER_DOT}{DISPLAY_SUFFIX})",
            "Target": f"%PREFIX%{TARGET}",
            "Arguments": "-m pydoc -b",
            "Icon": r"%PREFIX%Lib\idlelib\Icons\idle.ico",
            "IconIndex": 0,
        })
        if STD_PEP514:
            STD_PEP514[0]["InstallPath"]["IdlePath"] = f"%PREFIX%Lib\\idlelib\\idle.pyw"

    if ns.include_html_doc:
        STD_PEP514[0]["Help"]["Main Python Documentation"] = {
            "_": rf"%PREFIX%Doc\html\index.html",
        }
        STD_START[0]["Items"].append({
            "Name": f"{DISPLAY_NAME} {VER_DOT} Manuals{DISPLAY_SUFFIX}",
            "Target": r"%PREFIX%Doc\html\index.html",
        })
    elif ns.include_chm:
        STD_PEP514[0]["Help"]["Main Python Documentation"] = {
            "_": rf"%PREFIX%Doc\{PYTHON_CHM_NAME}",
        }
        STD_START[0]["Items"].append({
            "Name": f"{DISPLAY_NAME} {VER_DOT} Manuals{DISPLAY_SUFFIX}",
            "Target": "%WINDIR%hhc.exe",
            "Arguments": rf"%PREFIX%Doc\{PYTHON_CHM_NAME}",
        })

    STD_UNINSTALL.append({
        "kind": "uninstall",
        # Other settings will pick up sensible defaults
        "Publisher": "Python Software Foundation",
        "HelpLink": f"https://docs.python.org/{VER_DOT}/",
    })

    data = {
        "schema": 1,
        "id": f"{COMPANY.lower()}-{ID_TAG}",
        "sort-version": FULL_VERSION,
        "company": COMPANY,
        "tag": DISPLAY_TAG,
        "install-for": _not_empty(INSTALL_TAGS),
        "run-for": _not_empty(STD_RUN_FOR, "tag"),
        "alias": _not_empty(STD_ALIAS, "name"),
        "shortcuts": [
            *STD_PEP514,
            *STD_START,
            *STD_UNINSTALL,
        ],
        "display-name": f"{DISPLAY_NAME} {DISPLAY_VERSION}",
        "executable": rf".\{TARGET}",
        "url": f"{URL_BASE}{XYZ_VERSION}/{FILE_PREFIX}{FULL_VERSION}{FILE_SUFFIX}.zip"
    }

    return data